# EHR Data Quality & Interoperability Audit

Audits 555 synthetic patients' FHIR R4 records (Synthea's official sample dataset) for the issues that matter most when systems try to exchange EHR data: missing/incomplete fields, inconsistent use of standard coding systems (SNOMED-CT, LOINC, RxNorm, CVX), and broken references between resources (e.g. a condition pointing at an encounter that doesn't exist in our data).

**Data source:** `data/processed/*.csv`, produced by `parse_fhir_bundles.py` from the raw FHIR bundles in `data/raw_fhir/fhir/`.

**Questions this notebook answers:**
1. How complete is each table: which fields are missing, and how often?
2. Which coding systems are actually used across the clinical tables, and is that consistent?
3. Are there resources with more than one coding system attached (a common real-world interoperability pattern)?
4. Is referential integrity intact: do all patient/encounter references point to records that exist?
5. Are there duplicate resource IDs anywhere?

In [1]:
import pandas as pd

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

In [2]:
DATA_DIR = '../data/processed'

tables = {
    'patients': pd.read_csv(f'{DATA_DIR}/patients.csv'),
    'encounters': pd.read_csv(f'{DATA_DIR}/encounters.csv'),
    'conditions': pd.read_csv(f'{DATA_DIR}/conditions.csv'),
    'observations': pd.read_csv(f'{DATA_DIR}/observations.csv'),
    'medication_requests': pd.read_csv(f'{DATA_DIR}/medication_requests.csv'),
    'immunizations': pd.read_csv(f'{DATA_DIR}/immunizations.csv'),
    'procedures': pd.read_csv(f'{DATA_DIR}/procedures.csv'),
    'allergies': pd.read_csv(f'{DATA_DIR}/allergies.csv'),
}

for name, df in tables.items():
    print(f'{name:22s} {len(df):>8,} rows  {len(df.columns):>2} columns')

patients                    555 rows   8 columns
encounters               27,812 rows  10 columns
conditions               17,253 rows  10 columns
observations            131,703 rows  11 columns
medication_requests      24,256 rows   9 columns
immunizations             8,100 rows   9 columns
procedures               38,528 rows   9 columns
allergies                   499 rows   7 columns


### 1. Completeness - null rate by column, per table

In [3]:
for name, df in tables.items():
    null_pct = (df.isna().sum() / len(df) * 100).round(1)
    null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
    print(f'\n--- {name} ---')
    if null_pct.empty:
        print('No missing values in any column.')
    else:
        print(null_pct.to_string())


--- patients ---
No missing values in any column.

--- encounters ---
No missing values in any column.

--- conditions ---
abatement_date   21.9

--- observations ---
value   15.5
unit    15.5

--- medication_requests ---
code_system    1.5
code           1.5
code_display   1.5

--- immunizations ---
No missing values in any column.

--- procedures ---
No missing values in any column.

--- allergies ---
No missing values in any column.


## 2. Coding system coverage

For every clinical table (everything except `patients`), which coding system is actually
attached to each row's `code` field? 

In [4]:
clinical_tables = ['encounters', 'conditions', 'observations', 'medication_requests', 'immunizations', 'procedures', 'allergies']

for name in clinical_tables:
    df = tables[name]
    counts = df['code_system'].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(1)
    print(f'\n--- {name}: code_system distribution ---')
    print(pd.DataFrame({'rows': counts, '% of table': pct}).to_string())


--- encounters: code_system distribution ---
                         rows  % of table
code_system                              
http://snomed.info/sct  27812       100.0

--- conditions: code_system distribution ---
                         rows  % of table
code_system                              
http://snomed.info/sct  17253       100.0

--- observations: code_system distribution ---
                    rows  % of table
code_system                         
http://loinc.org  131703       100.0

--- medication_requests: code_system distribution ---
                                              rows  % of table
code_system                                                   
http://www.nlm.nih.gov/research/umls/rxnorm  23902        98.5
NaN                                            354         1.5

--- immunizations: code_system distribution ---
                             rows  % of table
code_system                                  
http://hl7.org/fhir/sid/cvx  8100       100.0

--

## 3. Multi-coded resources

In [5]:
for name in clinical_tables:
    df = tables[name]
    multi = (df['n_codings'] > 1).sum()
    print(f'{name:22s} {multi:>7,} / {len(df):>7,} rows ({multi / len(df) * 100:.1f}%) have more than one coding system attached')

encounters                   0 /  27,812 rows (0.0%) have more than one coding system attached
conditions                   0 /  17,253 rows (0.0%) have more than one coding system attached
observations               931 / 131,703 rows (0.7%) have more than one coding system attached
medication_requests          0 /  24,256 rows (0.0%) have more than one coding system attached
immunizations                0 /   8,100 rows (0.0%) have more than one coding system attached
procedures                   0 /  38,528 rows (0.0%) have more than one coding system attached
allergies                    0 /     499 rows (0.0%) have more than one coding system attached


## 4. Referential integrity

Does every `patient_id` referenced in a clinical table actually exist in `patients`?
Does every `encounter_id` referenced actually exist in `encounters`? 

In [6]:
valid_patient_ids = set(tables['patients']['patient_id'])
valid_encounter_ids = set(tables['encounters']['encounter_id'])

print('--- Orphaned patient_id references (patient not found in patients.csv) ---')
for name in clinical_tables:
    df = tables[name]
    orphaned = (~df['patient_id'].isin(valid_patient_ids)).sum()
    print(f'{name:22s} {orphaned:>7,} / {len(df):>7,} rows')

print('\n--- Missing or orphaned encounter_id references (no linked encounter) ---')
for name in ['conditions', 'observations', 'medication_requests', 'immunizations', 'procedures']:
    df = tables[name]
    missing = df['encounter_id'].isna().sum()
    orphaned = (~df['encounter_id'].isna() & ~df['encounter_id'].isin(valid_encounter_ids)).sum()
    print(f'{name:22s} missing: {missing:>7,}   orphaned (non-null but no match): {orphaned:>7,}   total rows: {len(df):>7,}')

--- Orphaned patient_id references (patient not found in patients.csv) ---
encounters                   0 /  27,812 rows
conditions                   0 /  17,253 rows
observations                 0 / 131,703 rows
medication_requests          0 /  24,256 rows
immunizations                0 /   8,100 rows
procedures                   0 /  38,528 rows
allergies                    0 /     499 rows

--- Missing or orphaned encounter_id references (no linked encounter) ---
conditions             missing:       0   orphaned (non-null but no match):       0   total rows:  17,253
observations           missing:       0   orphaned (non-null but no match):       0   total rows: 131,703
medication_requests    missing:       0   orphaned (non-null but no match):       0   total rows:  24,256
immunizations          missing:       0   orphaned (non-null but no match):       0   total rows:   8,100
procedures             missing:       0   orphaned (non-null but no match):       0   total rows:  38,52

## 5. Duplicate resource IDs

Each resource's `id` field should be unique within its table. Duplicates would indicate
either a parsing bug or a data quality issue in the source feed.

In [7]:
id_columns = {
    'patients': 'patient_id',
    'encounters': 'encounter_id',
    'conditions': 'condition_id',
    'observations': 'observation_id',
    'medication_requests': 'medication_request_id',
    'immunizations': 'immunization_id',
    'procedures': 'procedure_id',
    'allergies': 'allergy_id',
}

for name, id_col in id_columns.items():
    df = tables[name]
    dupes = df[id_col].duplicated().sum()
    print(f'{name:22s} {dupes:>7,} duplicate {id_col} value(s)')

patients                     0 duplicate patient_id value(s)
encounters                   0 duplicate encounter_id value(s)
conditions                   0 duplicate condition_id value(s)
observations                 0 duplicate observation_id value(s)
medication_requests          0 duplicate medication_request_id value(s)
immunizations                0 duplicate immunization_id value(s)
procedures                   0 duplicate procedure_id value(s)
allergies                    0 duplicate allergy_id value(s)


## Key findings

- Only three fields have real gaps. `abatement_date` is missing on 21.9% of
  conditions, which makes sense since many conditions are chronic and never resolve.
  `value`/`unit` are missing on 15.5% of observations because those are
  qualitative or coded results, not numeric ones. `medication_requests` is missing
  a coding system on 1.5% of rows (354 of 24,256): the one big issue in the dataset.
  
- Coding systems are used consistently across resource types: encounters, conditions,
  and procedures are 100% SNOMED-CT, observations are 100% LOINC, immunizations are
  100% CVX. Medication requests are 98.5% RxNorm. Allergies split between SNOMED-CT
  and RxNorm, which is expected since an allergy can be coded as a finding or as a
  specific substance.

  
- Multiple coding systems on the same record only show up in observations, and even
  there it's just 0.7% of rows. Every other resource type is single coded 100% of
  the time.

  
- No orphaned patient or encounter references across roughly 248,000 clinical
  records, and no duplicate IDs in any of the 8 tables.

The dataset is standards-consistent overall. The medication coding gap (354 rows) is
the only finding worth following up on in a real system.